# Claude Agent SDK Quickstart: an interactive walkthrough

In this notebook you will build an AI agent that **reads buggy code, finds the bugs, and fixes them on its own**, and you'll understand every line along the way.

**Plan**
1. Prerequisites and setup
2. Create a file with intentional bugs and *see* them fail
3. Understand the three parts of the agent code: `query`, `prompt`, `options`
4. Run the agent and watch the message stream
5. Verify the fix
6. Explore messages, tools, permission modes and customizations
7. Try your own prompts and troubleshoot

> Run the cells top to bottom. Cells that call the agent use your API key and cost a small amount of tokens.

## 1. Prerequisites

- **Python 3.10+**
- An **Anthropic API key** from the [Claude Console](https://platform.claude.com/)

Let's check your Python version.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"
print("Python version OK")

## 2. Install the SDK

`claude-agent-sdk` bundles a native Claude Code binary, so usually nothing else needs installing. The SDK is what actually runs the *agentic loop* (call Claude, run tools, feed results back, repeat).

In [ ]:
%pip install -q claude-agent-sdk python-dotenv

## 3. Load your API key from a `.env` file

The SDK reads `ANTHROPIC_API_KEY` from the environment of the process running your code. It does **not** load `.env` files automatically, so we load it ourselves with `python-dotenv`.

**Before running the next cell**, create a file named `.env` in the same folder as this notebook containing one line:

```
ANTHROPIC_API_KEY=your-api-key
```

(No quotes and no spaces around `=`. Add `.env` to your `.gitignore` so the key is never committed.)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# find_dotenv() searches the current folder, then parent folders
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )

print("Loaded .env from:", env_path or "(none found, using existing environment)")
print("API key loaded:", key[:7] + "..." + key[-4:])   # masked, never prints the full key

> Other providers are supported through environment variables, e.g. `CLAUDE_CODE_USE_BEDROCK=1` (Amazon Bedrock), `CLAUDE_CODE_USE_VERTEX=1` (Google Cloud), `CLAUDE_CODE_USE_FOUNDRY=1` (Microsoft Foundry).

## 4. Create a project folder with buggy code

The agent can access files in its **working directory** and subdirectories. We'll make a dedicated `my-agent` folder so the agent can't touch anything else.

`utils.py` has two intentional bugs:
1. `calculate_average([])` → division by zero
2. `get_user_name(None)` → `TypeError`

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("my-agent").resolve()
PROJECT_DIR.mkdir(exist_ok=True)

BUGGY_CODE = """def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)


def get_user_name(user):
    return user["name"].upper()
"""

def reset_buggy_file():
    """Restore utils.py to its original buggy state (handy for re-running the demo)."""
    (PROJECT_DIR / "utils.py").write_text(BUGGY_CODE)
    print("utils.py reset at", PROJECT_DIR / "utils.py")

reset_buggy_file()
print((PROJECT_DIR / "utils.py").read_text())

### See the bugs for yourself

Before asking an agent to fix anything, let's prove the bugs exist. We import the file and call each function with the bad input.

In [ ]:
import importlib, sys, traceback

sys.path.insert(0, str(PROJECT_DIR))

def load_utils():
    """Import (or re-import) utils.py so we always test the current file on disk."""
    import utils
    return importlib.reload(utils)

utils = load_utils()

for label, call in [
    ("calculate_average([])", lambda: utils.calculate_average([])),
    ("get_user_name(None)",   lambda: utils.get_user_name(None)),
]:
    try:
        print(label, "->", call())
    except Exception as e:
        print(f"{label} crashed: {type(e).__name__}: {e}")

## 5. The agent code, piece by piece

The whole agent is basically one function call:

```python
async for message in query(prompt=..., options=...):
    ...
```

### 5a. `query`: the entry point
`query()` starts the **agentic loop**. It returns an *async iterator*, so we use `async for` to receive messages as Claude works: its reasoning, tool calls, tool results, and the final outcome.

### 5b. `prompt`: what you want done
Plain language. Claude decides which tools to use.

### 5c. `options`: how the agent behaves
`ClaudeAgentOptions` configures the agent. The ones used here:

| Option | Meaning |
|---|---|
| `allowed_tools` | Tools that are **pre-approved** (no permission prompt). |
| `permission_mode="acceptEdits"` | Automatically approve file edits. |

Others you may use later: `system_prompt`, `mcp_servers`, `cwd`, and more.

Let's build the options object and inspect it.

In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
)

options = ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Glob"],  # pre-approved tools
    permission_mode="acceptEdits",           # auto-approve file edits
    cwd=str(PROJECT_DIR),                    # confine the agent to our folder
)

print("allowed_tools:  ", options.allowed_tools)
print("permission_mode:", options.permission_mode)
print("cwd:            ", options.cwd)

**What each tool does**

- `Read`: read a file's contents
- `Edit`: modify a file (exact text replacement)
- `Glob`: find files by pattern (e.g. `*.py`)

We set `cwd` so the agent works inside `my-agent/`. (The original quickstart simply runs from inside that folder.)

## 6. Run the agent

### A note on `asyncio.run`
In a normal script you'd end with `asyncio.run(main())`. **Jupyter already runs an event loop**, so calling `asyncio.run` here would error. Instead we define `async def main()` and simply write `await main()` in a cell.

### What the message loop does
- `AssistantMessage` holds a list of *content blocks*. A block with `.text` is Claude's reasoning; a block with `.name` is a tool call.
- `ResultMessage` is the final outcome. Its `subtype` is `"success"` when things went well.
- Other messages (system init, tool results) are filtered out to keep output readable.

In [ ]:
async def main():
    # Agentic loop: streams messages as Claude works
    async for message in query(
        prompt="Review utils.py for bugs that would cause crashes. Fix any issues you find.",
        options=options,
    ):
        # Print human-readable output
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text"):
                    print(block.text)              # Claude's reasoning
                elif hasattr(block, "name"):
                    print(f"Tool: {block.name}")   # Tool being called
        elif isinstance(message, ResultMessage):
            print(f"Done: {message.subtype}")      # Final result

await main()

## 7. Verify the fix

The agent should have: **Read** `utils.py` → **Analyzed** the edge cases → **Edited** the file with defensive code. Look at the new file and re-run the failing calls.

In [ ]:
print((PROJECT_DIR / "utils.py").read_text())

In [ ]:
utils = load_utils()

for label, call in [
    ("calculate_average([])",        lambda: utils.calculate_average([])),
    ("calculate_average([1, 2, 3])", lambda: utils.calculate_average([1, 2, 3])),
    ("get_user_name(None)",          lambda: utils.get_user_name(None)),
    ("get_user_name({'name': 'ada'})", lambda: utils.get_user_name({"name": "ada"})),
]:
    try:
        print(f"{label} -> {call()!r}")
    except Exception as e:
        print(f"{label} STILL crashes: {type(e).__name__}: {e}")

The exact fix can differ run to run (returning `0`, `None`, or raising a clear error are all valid choices), but nothing should crash unexpectedly.

## 8. Look under the hood: the raw message stream

The quickstart filters messages for readability. Without the filter you see **everything**, which is great for debugging. Let's reset the bug and print the *type* of every message.

In [ ]:
reset_buggy_file()

async def show_raw_messages():
    async for message in query(
        prompt="Review utils.py for bugs that would cause crashes. Fix any issues you find.",
        options=options,
    ):
        print(type(message).__name__)

await show_raw_messages()

You'll typically see a system init message, several `AssistantMessage`s (reasoning + tool calls), `UserMessage`s carrying tool results, and a final `ResultMessage`.

Want to see one message in full? Capture them in a list:

In [ ]:
reset_buggy_file()

collected = []
async for message in query(
    prompt="Review utils.py for bugs that would cause crashes. Fix any issues you find.",
    options=options,
):
    collected.append(message)

print(f"{len(collected)} messages received\n")
print(collected[-1])   # the ResultMessage

> **Streaming vs. collecting:** streaming (`async for` + print) shows live progress. For background jobs or CI where you don't need live output, collecting all messages first (as above) is perfectly fine.

## 9. Tools control what the agent can do

| Tools | What the agent can do |
|---|---|
| `Read`, `Glob`, `Grep` | Read-only analysis |
| `Read`, `Edit`, `Glob` | Analyze **and** modify code |
| `Read`, `Edit`, `Bash`, `Glob`, `Grep` | Full automation (can run commands) |

**Permission modes** control human oversight, e.g. `acceptEdits` auto-approves file edits. Anything not pre-approved or covered by the mode needs permission. See the SDK docs on *Permissions* for the full evaluation order.

Let's make a small reusable helper so we can try different configurations easily.

In [ ]:
async def run_agent(prompt, allowed_tools=("Read", "Edit", "Glob"), **extra_options):
    """Run the agent with a prompt and tool set, printing a readable stream."""
    opts = ClaudeAgentOptions(
        allowed_tools=list(allowed_tools),
        permission_mode="acceptEdits",
        cwd=str(PROJECT_DIR),
        **extra_options,
    )
    async for message in query(prompt=prompt, options=opts):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text"):
                    print(block.text)
                elif hasattr(block, "name"):
                    print(f"[tool] {block.name}")
        elif isinstance(message, ResultMessage):
            print(f"\nDone: {message.subtype}")

### Experiment A: read-only agent (cannot edit)
With only `Read`/`Glob`/`Grep`, the agent can *report* bugs but not fix them.

In [ ]:
reset_buggy_file()
await run_agent(
    "Review utils.py and list any crash-causing bugs. Do not change files.",
    allowed_tools=("Read", "Glob", "Grep"),
)
print("\nFile unchanged?", (PROJECT_DIR / "utils.py").read_text() == BUGGY_CODE)

### Experiment B: custom system prompt
`system_prompt` shapes the agent's style and rules.

In [ ]:
reset_buggy_file()
await run_agent(
    "Review utils.py for crash bugs and fix them.",
    system_prompt="You are a senior Python developer. Always follow PEP 8 style guidelines.",
)
print((PROJECT_DIR / "utils.py").read_text())

### Experiment C: enable `Bash` so the agent can run code
With `Bash` the agent can write tests, run them, and fix failures: a full write → run → fix loop. This lets the agent execute commands in `my-agent/`, so only enable it when you're comfortable with that.

In [ ]:
reset_buggy_file()
await run_agent(
    "Write unit tests for utils.py in test_utils.py, run them with pytest or unittest, and fix any failures.",
    allowed_tools=("Read", "Edit", "Glob", "Bash", "Write"),
)
print(sorted(p.name for p in PROJECT_DIR.iterdir()))

### Experiment D: add web search
Add `"WebSearch"` to `allowed_tools` so the agent can look things up while it works (not needed for this task, but this is how you'd enable it):

```python
ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Glob", "WebSearch"],
    permission_mode="acceptEdits",
)
```

## 10. Try other prompts

Reset and try any of these (swap the prompt in the cell):

- `"Add docstrings to all functions in utils.py"`
- `"Add type hints to all functions in utils.py"`
- `"Create a README.md documenting the functions in utils.py"` (needs `Write` in `allowed_tools`)

In [ ]:
reset_buggy_file()
await run_agent(
    "Add type hints and docstrings to all functions in utils.py",
)
print((PROJECT_DIR / "utils.py").read_text())

## 11. Troubleshooting

| Symptom | Fix |
|---|---|
| `Not logged in` / `Invalid API key` | Re-run the API key cell; make sure `ANTHROPIC_API_KEY` is set in **this** kernel. |
| `asyncio.run() cannot be called from a running event loop` | You're in Jupyter. Use `await main()` instead. |
| CLI not found / fails to start | pip may have installed the source distribution (e.g. ARM64 Windows). Install Claude Code natively; the SDK finds it on your `PATH`. |
| Agent says it can't edit | Add `"Edit"` to `allowed_tools` and keep `permission_mode="acceptEdits"`. |
| Changes from earlier runs confuse results | Call `reset_buggy_file()` first. |

## 12. Where to go next

- **Configure your agent**: all options in one place
- **Permissions**: control what needs approval
- **Hooks**: run custom code before/after tool calls
- **Sessions**: multi-turn agents that remember context
- **MCP servers**: connect databases, browsers, APIs
- **Hosting**: Docker, cloud, CI/CD
- **Example agents**: https://github.com/anthropics/claude-agent-sdk-demos